# 03 - Camada Silver

## Objetivo

Aplicar os tratamentos de qualidade, padronizar tipos de dados, consolidar entidades e preparar as tabelas para consumo analítico.

###Dados df_legado_regioes_pipe

In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from dateutil.relativedelta import *
from pyspark.storagelevel import StorageLevel
from delta.tables import *
from pytz import timezone
from array import *

In [0]:
df_legado_regioes_pipe = (
    spark.read
    .option("header", "true")
    .option("delimiter", "|")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/legado_regioes_pipe.txt")
)
display(df_legado_regioes_pipe)

In [0]:
##Renomeando colunas

df_legado_regioes_pipe = (df_legado_regioes_pipe
    .withColumnRenamed("regional_code", "cod_regional")
    .withColumnRenamed("regional_name", "nome_regional")
    .withColumnRenamed("state", "sigla_UF")
    .withColumnRenamed("manager_name", "nome_gerente")
    .withColumnRenamed("active_flag", "flag_ativo")
)

display(df_legado_regioes_pipe)

In [0]:
#Por regra de negócio uma linha foi desconsiderada, uma vez que não contém registros válidos.

df_legado_regioes_pipe = df_legado_regioes_pipe.filter(col("regional_code") != "XX")

display(df_legado_regioes_pipe)

In [0]:
#Ajuste fonte caixa alta

from pyspark.sql.functions import upper, col

df_legado_regioes_pipe = (df_legado_regioes_pipe
     .withColumn("cod_regional", upper(col("cod_regional")))
     .withColumn("nome_regional", upper(col("nome_regional")))
     .withColumn("sigla_UF", upper(col("sigla_UF")))
     .withColumn("nome_gerente", upper(col("nome_gerente")))
    )
display(df_legado_regioes_pipe)

In [0]:
#Padronização nomes

df_legado_regioes_pipe = df_legado_regioes_pipe.withColumn("sigla_UF",
    when(col("sigla_UF") == "STA CATARINA", "SC")
    .when(col("sigla_UF") == "GO", "GO")
    .when(col("sigla_UF") == "SAO PAULO", "SP")
    .otherwise(col("sigla_UF"))
)

df_legado_regioes_pipe = df_legado_regioes_pipe.withColumn("cod_regional",
    when(col("cod_regional") == "S", "SUL")
    .otherwise(col("cod_regional"))
    )

df_legado_regioes_pipe = df_legado_regioes_pipe.withColumn("nome_regional",
    when(col("nome_regional") == "REGIÃO SUL", "SUL")
    .otherwise(col("nome_regional"))
    )


display(df_legado_regioes_pipe)

###Tabela Silver: silver_tb_regioes_pipe

In [0]:
df_legado_regioes_pipe.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_tb_regioes_pipe")

In [0]:
df_final = spark.table("workspace.default.silver_tb_regioes_pipe")
display(df_final)

###Dados erp_pedidos_itens_2025

In [0]:
df_pedidos_itens2025 = (
    spark.read
    .option("header", "true")
    .option("delimiter", ",")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/erp_pedidos_itens_2025.csv")
)
display(df_pedidos_itens2025)

In [0]:
##Renomeando colunas

df_pedidos_itens2025 = (df_pedidos_itens2025
    .withColumnRenamed("order_id", "order_id")
    .withColumnRenamed("item_seq", "item_sequencia")
    .withColumnRenamed("product_code", "codigo_produto")
    .withColumnRenamed("quantity", "quantidade")
    .withColumnRenamed("unit_price", "preco_unitario")
    .withColumnRenamed("total_item", "total_item")
    .withColumnRenamed("item_status", "item_status")
)

display(df_pedidos_itens2025)

In [0]:
##acerto do código order_id
from pyspark.sql.functions import col, regexp_replace
df_pedidos_itens2025 = df_pedidos_itens2025.withColumn(
    "order_id",
    regexp_replace(col("order_id"), "^[oO]", "0")
)
display(df_pedidos_itens2025)

In [0]:
##acerto do código produto
from pyspark.sql.functions import col, regexp_replace

df_pedidos_itens2025 = df_pedidos_itens2025.withColumn(
    "codigo_produto",
    regexp_replace(col("codigo_produto"), "^[p]", "P")
)
display(df_pedidos_itens2025)

In [0]:
#acerto quantidade
#por regra (informação da área de negócio) valores negativos devem ser alterados para positivos
from pyspark.sql.functions import abs, col
df_pedidos_itens2025 = df_pedidos_itens2025.withColumn("quantidade", abs(col("quantidade")))
display(df_pedidos_itens2025)

In [0]:
#substituindo "," por "." na coluna preco_unitario
#alterando shema da coluna preco_unitario para double

from pyspark.sql.functions import col, when, regexp_replace

df_pedidos_itens2025 = df_pedidos_itens2025.withColumn("preco_unitario", regexp_replace(col("preco_unitario"), ",", "."))

df_pedidos_itens2025 = df_pedidos_itens2025.withColumn("preco_unitario", col("preco_unitario").cast("double"))



In [0]:
#acerto total_itens
#por regra (informação da área de negócio), valores negativos devem ser corrigido para valores positivos
from pyspark.sql.functions import abs, col
df_pedidos_itens2025 = df_pedidos_itens2025.withColumn("total_item", abs(col("total_item")))
display(df_pedidos_itens2025)

In [0]:
#Para identificar a quantidade foi informado pela área de negócio o racional: quantidade = (total_item/preco_unitario)

df_pedidos_itens2025 = df_pedidos_itens2025.withColumn("quantidade",
    when(
        col("quantidade").isNull() &
        col("preco_unitario").isNotNull() &
        (col("preco_unitario") != 0),
        col("total_item") / col("preco_unitario")
    ).otherwise(col("quantidade"))
)
display(df_pedidos_itens2025)

In [0]:
##acerto coluna item_status
##Por regra de negócio campo null deve ser considerado como "sem status"

df_pedidos_itens2025 = df_pedidos_itens2025.fillna({
    "item_status": "Sem Status"
    })

##ajuste na fonte
from pyspark.sql.functions import upper, col

df_pedidos_itens2025 = (df_pedidos_itens2025
     .withColumn("item_status", upper(col("item_status"))))

display(df_pedidos_itens2025)


###Tabela Silver: silver_tb_pedidos_itens2025

In [0]:
df_pedidos_itens2025.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_tb_pedidos_itens2025")

In [0]:
df_final2 = spark.table("workspace.default.silver_tb_pedidos_itens2025")
display(df_final2)

###Dados erp_pedidos_cabecalho_2025

In [0]:
df_pedidos_cabecalho2025 = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "true")
    .csv("/Workspace/case_data_engineer/sources/erp_pedidos_cabecalho_2025.csv")
)
display(df_pedidos_cabecalho2025)

In [0]:
##acerto do código customer_code

from pyspark.sql.functions import col, regexp_replace

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "customer_code",
    regexp_replace(col("customer_code"), "^[cC]", "C")
)

display(df_pedidos_cabecalho2025.select("customer_code"))


In [0]:
##acerto do código order_id

from pyspark.sql.functions import col, regexp_replace

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "order_id",
    regexp_replace(col("order_id"), "^[oO]", "0")
)

display(df_pedidos_cabecalho2025.select("order_id"))

In [0]:
from pyspark.sql.functions import countDistinct

df_pedidos_cabecalho2025.select(
    countDistinct("order_id").alias("qtd_distintos")
).show()

In [0]:
# identificando duplicados

from pyspark.sql.functions import count

df_pedidos_cabecalho2025.groupBy("order_id") \
  .count() \
  .filter("count > 1") \
  .show()

In [0]:
#pradronizar nomes da coluna status_order
display(df_pedidos_cabecalho2025.groupBy("status_order").count())

In [0]:
##Por regra de negócio campo null deve ser considerado como "sem status"

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.fillna({
    "status_order": "SEM STATUS"
    })

##ajuste na fonte e nomes

from pyspark.sql.functions import upper, col, when


df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "status_order",
    when(col("status_order").isin("Faturado", "faturado"), "FATURADO")
    .when(col("status_order").isin("EM_SEPARACAO", "em separacao"), "EM SEPARAÇÃO")
    .when(col("status_order") == "entregue", "ENTREGUE")
    .when(col("status_order") == "cancelado", "CANCELADO")
    .otherwise(col("status_order"))
)

display(df_pedidos_cabecalho2025.groupBy("status_order").count())

In [0]:
##identificando os tipos de canais e prioridades do pedido

display(df_pedidos_cabecalho2025.select("payment_details"))

In [0]:
#ajuste coluna payment_details

from pyspark.sql.functions import col, regexp_replace, from_json
from pyspark.sql.types import StructType, StructField, StringType

# Schema do JSON
schema = StructType([
    StructField("priority", StringType(), True),
    StructField("source", StringType(), True)
])

# Remover aspas externas e trocar "" por "

df_pedidos_cabecalho2025 = (df_pedidos_cabecalho2025
    .withColumn(
        "payment_details_json",
        regexp_replace(
            regexp_replace(col("payment_details"), '^"|"$', ""),
            '""',
            '"'
        )
    )
    .withColumn(
        "payment_details_json",
        from_json(col("payment_details_json"), schema)
    )
    .withColumn("priority", col("payment_details_json.priority"))
    .withColumn("source", col("payment_details_json.source"))
    .drop("payment_details_json")
)

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.drop("payment_details")
display(df_pedidos_cabecalho2025)



In [0]:
#ajuste nos nomes das colunas priority e source

display(df_pedidos_cabecalho2025.groupBy("priority").count())
display(df_pedidos_cabecalho2025.groupBy("source").count())

In [0]:
df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "priority",
    when(col("priority").isin("high"), "ALTO")
    .when(col("priority").isin("medium"), "MÉDIO")
    .when(col("priority").isin("low"), "BAIXO")
      .otherwise(col("priority"))
)

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "source",
    when(col("source").isin("Portal"), "PORTAL")
    .when(col("source").isin("erp"), "ERP")
    .when(col("source").isin("app"), "APP")
      .otherwise(col("source"))
)

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.fillna({
    "source": "SEM STATUS",
    "priority": "SEM STATUS"
    })

display(df_pedidos_cabecalho2025.groupBy("priority").count())
display(df_pedidos_cabecalho2025.groupBy("source").count())

In [0]:
#Por regra de negócio N/A na coluna gross_amount deve ser desconsiderado da base de dados.
df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.filter(col("gross_amount") != "N/A")

In [0]:
#substituindo "," por "." na coluna gross_amount

#alterando shema da coluna gross_amount para double

from pyspark.sql.functions import when, col, regexp_replace

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "gross_amount",
    regexp_replace(col("gross_amount"), ",", ".")
).withColumn(
    "gross_amount",
    col("gross_amount").cast("double")
    )

display(df_pedidos_cabecalho2025.select("gross_amount"))

In [0]:
#por regra de negócio, valores negativos devem ser corrigido para valores positivos na coluna net_amount

from pyspark.sql.functions import abs, col
df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn("net_amount", abs(col("net_amount")))
display(df_pedidos_cabecalho2025.select("net_amount"))

In [0]:
#identificando e tratando a coluna order_date

df_pedidos_cabecalho2025.filter(col("order_date").isNotNull()).select("order_date").show(20, False)

In [0]:
from pyspark.sql.functions import col, coalesce, expr

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "order_date",
    coalesce(
        expr("try_to_date(cast(order_date as string), 'yyyy-MM-dd')"),
        expr("try_to_date(cast(order_date as string), 'yyyy/MM/dd')"),
        expr("try_to_date(cast(order_date as string), 'dd/MM/yyyy')")
    )
)
display(df_pedidos_cabecalho2025)

In [0]:
#tratando a coluna promised_date
from pyspark.sql.functions import col, coalesce, expr

df_pedidos_cabecalho2025 = df_pedidos_cabecalho2025.withColumn(
    "promised_date",
    coalesce(
        expr("try_to_date(cast(promised_date as string), 'yyyy-MM-dd')"),
        expr("try_to_date(cast(promised_date as string), 'yyyy/MM/dd')"),
        expr("try_to_date(cast(promised_date as string), 'dd/MM/yyyy')")
    )
)
display(df_pedidos_cabecalho2025)

In [0]:
#ajuste nome colunas

df_pedidos_cabecalho2025 = (df_pedidos_cabecalho2025
    .withColumnRenamed("order_id", "order_id")
    .withColumnRenamed("customer_code", "codigo_cliente")
    .withColumnRenamed("seller_id", "id_vendedor")
    .withColumnRenamed("order_date", "data_pedido")
    .withColumnRenamed("promised_date", "data_prometida")
    .withColumnRenamed("status_order", "status_pedido")
    .withColumnRenamed("gross_amount", "valor_bruto")
    .withColumnRenamed("discount_amount", "valor_desconto")
    .withColumnRenamed("net_amount", "valor_liquido")
    .withColumnRenamed("priority", "prioridade")
    .withColumnRenamed("source", "canal")
    .withColumnRenamed("last_update", "ultima_atualizacao")
)

display(df_pedidos_cabecalho2025)

###Tabela Silver: silver_tb_pedidos_cabecalho2025

In [0]:
df_pedidos_cabecalho2025.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_tb_pedidos_cabecalhos2025")

In [0]:
df_final3 = spark.table("workspace.default.silver_tb_pedidos_cabecalhos2025")
display(df_final3)